In [1]:
!pip install -qU langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 8.7 MB/s eta 0:00:00


In [2]:
import os
from langchain.chat_models import init_chat_model
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
model = init_chat_model("gpt-4o-mini")

In [3]:
from langchain.tools import tool

#create booking hotel tool
@tool
def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."

#create flight hotel tool
@tool
def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport}."

In [4]:
from langchain.agents import create_agent

#create hotel agent
flight_assistant = create_agent(
    model=model,
    tools=[book_flight],
    system_prompt="You are a flight booking assistant",
    name="flight_assistant"
)

#create hotel agent
hotel_assistant = create_agent(
    model=model,
    tools=[book_hotel],
    system_prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)

In [5]:
flight_assistant.invoke({"messages": [{"role": "user", "content": "Book a flight from CGK to CDG."}]})

{'messages': [HumanMessage(content='Book a flight from CGK to CDG.', additional_kwargs={}, response_metadata={}, id='d9b23947-4c66-4bf5-b6d1-aa3558645b0e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 67, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DWicBXsuMfsiJanKods7EzWQJVfns', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='flight_assistant', id='lc_run--019daaff-875b-7a33-8363-28975319ef6d-0', tool_calls=[{'name': 'book_flight', 'args': {'from_airport': 'CGK', 'to_airport': 'CDG'}, 'id': 'call_LzGAwtr9QRJ4SlSotNa8UVAx', 'type': 'tool_call'}], invalid_tool_

In [6]:
@tool
def flight_agent(from_airport: str, to_airport: str):
    """Book a flight"""
    return flight_assistant.invoke({"messages": [{"role": "user", "content": f"Book a flight from {from_airport} to {to_airport}."}]})

@tool
def hotel_agent(hotel_name: str):
    """Book a hotel"""
    return hotel_assistant.invoke({"messages": [{"role": "user", "content": f"Book a stay at {hotel_name}."}]})

In [7]:
supervisor = create_agent(
    tools=[flight_agent, hotel_agent], #list of agents that will helps supervisor agent
    model=model, #model for supervisor agent
    system_prompt= "You manage a hotel booking assistant and a flight booking assistant. Assign work to them." #prompt for supervisor agent
)

In [13]:
query = (
    "book a flight from BOS to JFK and a stay at McKittrick Hotel."
)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  flight_agent (call_bvLkmUK5xeEyKlnabmO92CYf)
 Call ID: call_bvLkmUK5xeEyKlnabmO92CYf
  Args:
    from_airport: BOS
    to_airport: JFK
  hotel_agent (call_RhpsxoqzSWUXFJ3LWVOOLHCr)
 Call ID: call_RhpsxoqzSWUXFJ3LWVOOLHCr
  Args:
    hotel_name: McKittrick Hotel
================================= Tool Message =================================
Name: hotel_agent

{'messages': [HumanMessage(content='Book a stay at McKittrick Hotel.', additional_kwargs={}, response_metadata={}, id='080d193f-d5fe-41fd-b929-e64d02925d9b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 60, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 

In [14]:
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": query}]}
)
answer = response["messages"][-1].content
print(answer)

Your bookings have been successfully completed:

- **Flight**: You have booked a flight from Boston (BOS) to New York (JFK).
- **Hotel Stay**: Your stay at the McKittrick Hotel has been successfully booked.

If you need any further assistance or details, feel free to ask!


In [15]:
response["messages"]

[HumanMessage(content='book a flight from BOS to JFK and a stay at McKittrick Hotel.', additional_kwargs={}, response_metadata={}, id='ee18f63c-383a-4c55-82f7-957d8c4b17dd'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 102, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4431c98ba', 'id': 'chatcmpl-DWivn1bLFTpyb0NiYEL7tctzQfqpt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dab12-1a5f-75c2-849b-3cb9348099de-0', tool_calls=[{'name': 'flight_agent', 'args': {'from_airport': 'BOS', 'to_airport': 'JFK'}, 'id': 'call_c7yADJX3SplwGXuCC8NeFc7V', 'type': 'tool_call'}, {'name': 'hotel_age